# V1: Static Multi-Agent Research Workflow

This stable tutorial implements a pure-Python deep research workflow using a
structured Planner, deterministic Executor, tool-using Research Agent, and
Draft-Critic-Revision writing pipeline. It corresponds to Git tag `v0.1.0`.

Continue with `02_v2_adaptive_deep_research.ipynb` to study adaptive
supervision, isolated research workers, compression, and parallel research.

# 0. Setup

## 0.1 Imports

In [ ]:
import importlib
import json
import os
import platform
import re
import sys
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

import agent_schema
from agent_tools import (
    arxiv_search_tool,
    arxiv_tool_def,
    tavily_search_tool,
    tavily_tool_def,
)

## 0.2 Runtime Configuration

In [ ]:
EXPECTED_PYTHON_VERSION = (3, 12)
PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / ".env"

if sys.version_info[:2] != EXPECTED_PYTHON_VERSION:
    raise RuntimeError(
        "Python 3.12 is required. Start Jupyter with `uv run jupyter lab`."
    )

load_dotenv(dotenv_path=ENV_PATH, override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")

if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not configured in .env.")

if not MODEL_NAME:
    raise RuntimeError("MODEL_NAME is not configured in .env.")

client = OpenAI(api_key=openai_api_key)

agent_schema = importlib.reload(agent_schema)
PlanStep = agent_schema.PlanStep
TaskPlan = agent_schema.TaskPlan

print(f"[SETUP] Python: {platform.python_version()}")
print(f"[SETUP] Executable: {sys.executable}")
print(f"[SETUP] Project directory: {PROJECT_ROOT}")
print(f"[SETUP] .env found: {ENV_PATH.exists()}")
print(f"[SETUP] Model: {MODEL_NAME}")
print(f"[SETUP] Tavily configured: {bool(os.getenv('TAVILY_API_KEY'))}")

## 0.3 Notebook Display Helper

In [ ]:
def show(title: str, content: str) -> None:
    display(Markdown(f"## {title}\n\n{content}"))

# 1. OpenAI Response API

In [ ]:
# response = client.responses.create(
#     model=MODEL_NAME,
#     instructions="You are a concise technical writer.",
#     input="Write a research report on agentic design patterns."
# )

In [ ]:
# Markdown(response.output_text)

# 2. Reflection Design Pattern

In [ ]:
# def call_model(instructions: str, input_text: str) -> str:
#     response = client.responses.create(
#         model=MODEL_NAME,
#         instructions=instructions,
#         input=input_text
#     )
#     return response.output_text

In [ ]:
# task = """
# Write a research report on agentic design patterns.
# """

# draft = call_model(
#     instructions="""
#     You are an essay writer.

#     Given a topic: {topic}, output an essay on that topic.

#     You need to output the complete essay.

#     The length of the essay should longer than 500 words.

#     Do not output other unrelated content.
#     """,
#     input_text=task
# )

# show("Draft", draft)

In [ ]:
# critique = call_model(
#     instructions="""
#     You are a strict but constructive critic.

#     Evaluate the draft for:
#     1. correctness
#     2. clarity
#     3. completeness
#     4. unsupported claims

#     Identify specific problems and suggest improvements.
#     Do not rewrite the draft.
#     """,
#     input_text=f"""
#     Original task:
#     {task}

#     Draft:
#     {draft}
#     """
# )

# show("Critique", critique)

In [ ]:
# revision = call_model(
#     instructions="""
#     You are a technical editor.

#     Revise the draft using the useful parts of the critique.
#     Preserve correct content, fix identified problems, and
#     return only the improved answer.
#     """,
#     input_text=f"""
#     Original task:
#     {task}

#     Draft:
#     {draft}

#     Critique:
#     {critique}
#     """,
# )

# show("Revision", revision)

# 3. Tool Calling Design Pattern

In [ ]:
# # Test the arXiv search tool
# topic = "LLM"

# arxiv_results = arxiv_search_tool(topic, max_results=3)

# # Show formatted arxiv_results
# for i, paper in enumerate(arxiv_results, 1):
#     if "error" in paper:
#         print(f"❌ Error: {paper['error']}")
#     else:
#         print(f"📄 Paper {i}")
#         print(f"  Title     : {paper['title']}")
#         print(f"  Authors   : {', '.join(paper['authors'])}")
#         print(f"  Published : {paper['published']}")
#         print(f"  URL       : {paper['url']}\n")


# print("\n🧾 Raw arxiv_Results:\n")
# print(json.dumps(arxiv_results, indent=2))

In [ ]:
# # Test the Tavily search tool
# topic = "deeplearning.ai"

# tavily_results = tavily_search_tool(topic)
# for item in tavily_results:
#     print(item)

In [ ]:
TOOLS = [arxiv_tool_def, tavily_tool_def]

TOOL_REGISTRY = {
    "arxiv_search_tool": arxiv_search_tool,
    "tavily_search_tool": tavily_search_tool,
}

def execute_tool(name: str, arguments: dict) -> Any:
    tool = TOOL_REGISTRY.get(name)

    if tool is None:
        raise ValueError(f"Tool '{name}' not found in the registry.")

    return tool(**arguments)

In [ ]:
# TOOL_USE_INSTRUCTIONS = """
# You are a research assistant.

# Use tavily_search_tool for general web information and recent developments.
# Use arxiv_search_tool for academic papers and research literature.

# Base your answer on the retrieved evidence.
# Include source URLs when available.
# Do not invent sources or claim that a tool returned information it did not return.
# """

In [ ]:
# def tool_use_agent(user_input: str, max_steps: int = 8) -> str:
#     input_items: list[Any] = [
#         {
#             "role": "user",
#             "content": user_input
#         }
#     ]

#     for _ in range(max_steps):
#         response = client.responses.create(
#             model=MODEL_NAME,
#             instructions=TOOL_USE_INSTRUCTIONS,
#             input=input_items,
#             tools=TOOLS
#         )

#         input_items.extend(response.output)

#         tool_calls = [
#             item for item in response.output if item.type == "function_call"
#         ]

#         if not tool_calls:
#             return response.output_text

#         for tool_call in tool_calls:
#             try:
#                 arguments = json.loads(tool_call.arguments)

#                 result = execute_tool(name=tool_call.name, arguments=arguments)

#                 tool_output = {
#                     "ok": True,
#                     "result": result
#                 }

#             except Exception as e:
#                 tool_output = {
#                     "ok": False,
#                     "error": str(e)
#                 }

#             input_items.append({
#                 "type": "function_call_output",
#                 "call_id": tool_call.call_id,
#                 "output": json.dumps(tool_output, ensure_ascii=False),
#             })

#     raise RuntimeError("Max steps reached without producing a final answer.")


In [ ]:
# tool_use_result = tool_use_agent(
#     """
#     Research recent developments in agentic AI.
#     Find both general web information and relevant arXiv papers.
#     Summarize the main themes and include source URLs.
#     """
# )

# show("Tool Use Agent", tool_use_result)

# 4. Planning & Multi-agent Design Pattern

## 4.1 Planner Agent

In [ ]:
PLANNER_INSTRUCTIONS = """
You are the Planner Agent for a multi-agent deep research workflow.

Available agents and their exact identifiers:
- research_agent: searches the web and arXiv and returns source-backed notes.
- writer_agent: drafts and revises complete Markdown reports.
- editor_agent: reviews drafts and returns actionable feedback.

Create the smallest valid TaskPlan that satisfies the workflow constraints.
Assign exactly one available agent to every step.
Put the action, evidence requirements, and required result directly in each task.
Combine closely related research or writing work instead of creating many small steps.

Include research, one initial draft, exactly one editorial review, and one final revision.
Use at least one research_agent step before the first writer_agent step.
The editor_agent step must be followed immediately by the final writer_agent step.
The final writer_agent step must revise the draft and return the publication-ready report.
Do not add a separate polishing or finalization step after the final revision.
Include only research workflow steps that directly contribute to the final
source-backed report and can be executed by the assigned agents.
Leave report length to the runtime requirements supplied to writer_agent.
Do not execute the plan or include research findings in the plan.
""".strip()

In [ ]:
def validate_task_plan(plan: TaskPlan, max_steps: int) -> None:
    """Validate structural and workflow invariants for a research plan."""

    if max_steps < 4:
        raise ValueError("max_steps must be at least 4 for this workflow.")

    if not plan.steps:
        raise RuntimeError("Planner agent returned a plan with no steps.")

    if len(plan.steps) > max_steps:
        raise RuntimeError(
            f"Planner returned {len(plan.steps)} steps; maximum is {max_steps}."
        )

    expected_ids = list(range(1, len(plan.steps) + 1))
    actual_ids = [step.id for step in plan.steps]

    if actual_ids != expected_ids:
        raise RuntimeError(
            f"Planner step IDs must be consecutive and one-based: {actual_ids!r}"
        )

    route = [step.agent for step in plan.steps]
    research_positions = [
        index for index, agent in enumerate(route) if agent == "research_agent"
    ]
    writer_positions = [
        index for index, agent in enumerate(route) if agent == "writer_agent"
    ]
    editor_positions = [
        index for index, agent in enumerate(route) if agent == "editor_agent"
    ]

    if not research_positions:
        raise RuntimeError("Task plan must include at least one research_agent step.")

    if len(editor_positions) != 1:
        raise RuntimeError("Task plan must include exactly one editor_agent step.")

    editor_position = editor_positions[0]

    if not writer_positions or writer_positions[-1] != len(route) - 1:
        raise RuntimeError("The final plan step must use writer_agent.")

    if editor_position != len(route) - 2:
        raise RuntimeError(
            "The editor_agent step must be immediately before the final writer_agent step."
        )

    draft_writer_positions = [
        position for position in writer_positions if position < editor_position
    ]

    if not draft_writer_positions:
        raise RuntimeError("Task plan must draft the report before editorial review.")

    if not any(position < draft_writer_positions[0] for position in research_positions):
        raise RuntimeError("Research must occur before the initial report draft.")

def planner_agent(
    topic: str,
    model: str = MODEL_NAME,
    max_steps: int = 4,
    verbose: bool = True,
) -> TaskPlan:
    """
    Generate a structured execution plan for a research workflow.

    Args:
        topic: The research topic to investigate.
        model: The language model to use.
        max_steps: Maximum number of executable plan steps.
        verbose: Whether to print concise execution logs.

    Returns:
        A validated task plan with explicit agent assignments.
    """

    topic = topic.strip()

    if not topic:
        raise ValueError("Topic must not be empty.")

    if max_steps < 4:
        raise ValueError("max_steps must be at least 4 for this workflow.")

    if verbose:
        print("[PLANNER_AGENT] Starting planning.")
        print(f"[PLANNER_AGENT] Topic: {topic}")
        print("[PLANNER_AGENT] Model call #1")

    response = client.responses.parse(
        model=model,
        instructions=PLANNER_INSTRUCTIONS,
        input=(
            f"Create a research plan with 4 to {max_steps} steps "
            f"for the following topic:\n\n{topic}"
        ),
        text_format=TaskPlan,
    )

    plan = response.output_parsed

    if plan is None:
        raise RuntimeError(
            f"Planner agent did not return a valid TaskPlan. Raw output: {response.output_text!r}"
        )

    validate_task_plan(plan, max_steps=max_steps)

    if verbose:
        route = " -> ".join(step.agent for step in plan.steps)
        print("[PLANNER_AGENT] Planning completed.")
        print(f"[PLANNER_AGENT] Total steps: {len(plan.steps)}")
        print(f"[PLANNER_AGENT] Agent route: {route}")

    return plan

In [ ]:
# topic = "What are the recent developments in AI design patterns for improving the reliability of LLM agents?"
# plan = planner_agent(topic)
# print(plan.model_dump_json(indent=2))

## 4.2 Research Agent

In [ ]:
RESEARCH_INSTRUCTIONS = """
You are the Research Agent in a multi-agent deep research system.

Execute exactly one research step assigned by the Planner Agent.

Research requirements:

- Focus only on the supplied task step.
- Do not create or modify the task plan.
- Do not answer the user's overall question.
- Use the available tools to collect evidence.
- Use tavily_search_tool for recent developments, documentation, websites,
  articles, and general web information.
- Use arxiv_search_tool for papers, methods, experiments, and benchmarks.
- Use plain semantic keywords for arXiv. Do not use URLs, site operators,
  the word arXiv, or web-style date ranges such as 2020..2026.
- Refine a search query only when its results leave a material evidence gap.
- Prefer 3 to 5 focused searches over many overlapping searches.
- Do not repeat searches that differ only slightly.
- Base factual claims on tool results rather than unsupported memory.
- Do not invent sources, titles, authors, dates, or URLs.
- Preserve source URLs exactly as returned by the tools.
- Prefer primary and academic sources when available.
- Clearly identify incomplete or conflicting evidence.
- Stop once the supplied task can be answered with sufficient relevant evidence.
- After collecting several useful primary and web sources, synthesize the
  findings instead of continuing to search.

Return concise research notes in Markdown using this structure:

## Summary

A concise summary of the research result.

## Findings

Evidence-backed findings with source links.

## Sources

A list of source titles and URLs.

## Limitations

Missing, uncertain, or conflicting information.

Return only the structured research notes. Do not include conversational
text or a user-facing final answer.

Your output will be consumed by later agents through the execution history.
""".strip()

In [ ]:
def research_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    max_tool_rounds: int = 8,
    verbose: bool = True,
) -> str:
    if step.agent != "research_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to "
            f"{step.agent!r}, not 'research_agent'."
        )

    research_request = {
        "user_request": user_request,
        "task": step.task,
        "execution_history": history,
    }

    input_items: list[Any] = [
        {
            "role": "user",
            "content": json.dumps(
                research_request,
                ensure_ascii=False,
            ),
        }
    ]

    model_calls = 0
    tool_rounds = 0
    total_tool_calls = 0
    tool_counts: dict[str, int] = {}

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")

    while True:
        tool_budget_exhausted = tool_rounds >= max_tool_rounds
        request_input = input_items

        if tool_budget_exhausted:
            request_input = [
                *input_items,
                {
                    "role": "user",
                    "content": (
                        "The tool budget is exhausted. Do not request more tools. "
                        "Produce the final research notes using the evidence "
                        "already collected."
                    ),
                },
            ]

            if verbose:
                print(
                    f"[{step.agent.upper()}] Tool budget exhausted; "
                    "forcing final synthesis."
                )

        model_calls += 1

        if verbose:
            print(f"[{step.agent.upper()}] Model call #{model_calls}")

        response = client.responses.create(
            model=model,
            instructions=RESEARCH_INSTRUCTIONS,
            input=request_input,
            tools=TOOLS,
            tool_choice="none" if tool_budget_exhausted else "auto",
            parallel_tool_calls=False,
        )

        # Preserve all model output items for the next API request.
        input_items.extend(response.output)

        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # No tool call means the agent has produced its final research notes.
        if not tool_calls:
            research_result = response.output_text.strip()

            if not research_result:
                raise RuntimeError(
                    f"Research agent returned no result for step {step.id!r}."
                )

            if verbose:
                print(f"[{step.agent.upper()}] Research step {step.id} completed.")
                print(f"[{step.agent.upper()}] Total model calls: {model_calls}")
                print(f"[{step.agent.upper()}] Total tool rounds: {tool_rounds}")
                print(f"[{step.agent.upper()}] Total tool calls: {total_tool_calls}")
                print(f"[{step.agent.upper()}] Tool usage counts: {tool_counts}")
            return research_result

        for tool_call in tool_calls:

            total_tool_calls += 1
            tool_counts[tool_call.name] = (
                tool_counts.get(tool_call.name, 0) + 1
            )

            if verbose:
                print(
                    f"[{step.agent.upper()}] Tool call #{total_tool_calls}: "
                    f"{tool_call.name}, Arguments: {tool_call.arguments}"
                )

            try:
                arguments = json.loads(tool_call.arguments)

                result = execute_tool(
                    name=tool_call.name,
                    arguments=arguments,
                )

                if isinstance(result, list):
                    tool_errors = [
                        str(item["error"])
                        for item in result
                        if isinstance(item, dict) and "error" in item
                    ]
                    if tool_errors:
                        raise RuntimeError("; ".join(tool_errors))
                    result_summary = f"items={len(result)}"
                else:
                    result_summary = f"type={type(result).__name__}"

                tool_output = {
                    "ok": True,
                    "result": result,
                }

            except Exception as exc:  # noqa: BLE001
                tool_output = {
                    "ok": False,
                    "error": str(exc),
                }
                result_summary = f"error={type(exc).__name__}: {exc}"

            if verbose:
                print(
                    f"[{step.agent.upper()}] Tool result "
                    f"#{total_tool_calls}: {result_summary}"
                )

            input_items.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(
                        tool_output,
                        ensure_ascii=False,
                    ),
                }
            )

        tool_rounds += 1

In [ ]:
# research_step = next(step for step in plan.steps if step.agent == "research_agent")

# research_result = research_agent(
#     user_request=topic,
#     step=research_step,
#     history=[],
# )

# show("Research Agent Test", research_result)

## 4.3 Writer Agent

In [ ]:
WRITER_INSTRUCTIONS = """
You are the Writer Agent in a multi-agent deep research system.

Execute exactly one writing step assigned by the Planner Agent.

The supplied execution history may contain:

- Research notes from one or more research steps
- An earlier draft
- Editorial feedback

Writing requirements:

- Follow the supplied task exactly.
- Use only the information contained in the execution history.
- Do not invent facts, sources, titles, dates, or URLs.
- Preserve source URLs exactly as provided.
- Clearly distinguish established findings from uncertain conclusions.
- Synthesize overlapping research rather than repeating it.
- Resolve minor inconsistencies when the supplied evidence allows it.
- Preserve important limitations and disagreements between sources.
- Use descriptive headings and a logical narrative structure.
- Include citations close to the claims they support.
- Avoid discussing the internal agent workflow in the report.
- Do not describe an arXiv preprint as peer reviewed unless the evidence says so.

If the task requests an initial draft:

- Synthesize the supplied research results into a coherent report.
- Ensure the report addresses the original user request.

If the task requests a revision:

- Use both the earlier draft and the editorial feedback.
- Address all required editorial changes.
- Return the complete revised report, not a list of modifications.
- When report_requirements are provided, follow their word limits.
- Use one top-level Markdown title and second-level section headings.
- Include ## Abstract, ## Executive Summary, and ## References sections.
- Deduplicate references and preserve available source metadata.
- Do not include a change summary or editorial commentary.

Return only the requested report content in Markdown.
Do not include a preface, afterword, change summary, follow-up question,
offer to continue, or any conversational text addressed to the user.
When producing the final report, end the document with the References section.
""".strip()

In [ ]:
def writer_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    report_requirements: dict[str, int] | None = None,
    verbose: bool = True,
) -> str:
    if step.agent != "writer_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to "
            f"{step.agent!r}, not 'writer_agent'."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")
        print(f"[{step.agent.upper()}] Context steps: {len(history)}")
        print(f"[{step.agent.upper()}] Model call #1")

    writer_request = {
        "user_request": user_request,
        "task_step": step.model_dump(mode="json"),
        "execution_history": history,
        "report_requirements": report_requirements,
    }

    response = client.responses.create(
        model=model,
        instructions=WRITER_INSTRUCTIONS,
        input=json.dumps(
            writer_request,
            ensure_ascii=False,
        ),
    )

    result = response.output_text.strip()

    if not result:
        raise RuntimeError(
            f"Writer agent returned no result for step {step.id!r}."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Writing step {step.id} completed.")
        print(f"[{step.agent.upper()}] Output characters: {len(result)}")

    return result

In [ ]:
# writer_step = next(
#     step
#     for step in plan.steps
#     if step.agent == "writer_agent"
# )

# writer_history = [
#     {
#         "step_id": research_step.id,
#         "agent": research_step.agent,
#         "task": research_step.task,
#         "result": research_result,
#     }
# ]

# writer_result = writer_agent(
#     user_request=topic,
#     step=writer_step,
#     history=writer_history,
# )

# show("Writer Agent Test", writer_result)

## 4.4 Editor Agent

In [ ]:
EDITOR_INSTRUCTIONS = """
You are the Editor Agent in a multi-agent deep research system.

Critically review exactly one draft according to the step assigned by the
Planner Agent.

Review requirements:

- Evaluate whether the draft addresses the original user request.
- Evaluate factual support using only the supplied content.
- Identify unsupported or overstated claims.
- Check whether citations are placed near the claims they support.
- Identify missing citations, suspicious citations, and broken logical links.
- Check coverage, structure, clarity, concision, and internal consistency.
- Identify important research findings that were omitted or distorted.
- Distinguish required corrections from optional improvements.
- Do not invent facts or sources.
- Do not rewrite the entire report.
- Make every criticism specific and actionable.
- Keep the review concise and prioritize no more than ten required changes.

Return the review in Markdown using this structure:

## Verdict

Use either APPROVE or REVISE and provide a short explanation.

## Strengths

List the strongest aspects of the draft.

## Required Changes

List factual, structural, or coverage problems that must be fixed.

## Citation Issues

List unsupported claims and citation problems.

## Optional Improvements

List non-essential improvements.

Return only the structured editorial review. Do not include a rewritten
report or conversational text.

Your review will be consumed by the Writer Agent during revision.
""".strip()

In [ ]:
def editor_agent(
    user_request: str,
    step: PlanStep,
    history: list[dict[str, Any]],
    model: str = MODEL_NAME,
    verbose: bool = True,
) -> str:
    if step.agent != "editor_agent":
        raise ValueError(
            f"Task step {step.id!r} is assigned to "
            f"{step.agent!r}, not 'editor_agent'."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Starting step {step.id}")
        print(f"[{step.agent.upper()}] Task: {step.task}")
        print(f"[{step.agent.upper()}] Context steps: {len(history)}")
        print(f"[{step.agent.upper()}] Model call #1")

    editor_request = {
        "user_request": user_request,
        "task_step": step.model_dump(mode="json"),
        "execution_history": history,
    }

    response = client.responses.create(
        model=model,
        instructions=EDITOR_INSTRUCTIONS,
        input=json.dumps(
            editor_request,
            ensure_ascii=False,
        ),
    )

    result = response.output_text.strip()

    if not result:
        raise RuntimeError(
            f"Editor agent returned no result for step {step.id!r}."
        )

    if verbose:
        print(f"[{step.agent.upper()}] Editing step {step.id} completed.")
        print(f"[{step.agent.upper()}] Output characters: {len(result)}")

    return result

In [ ]:
# editor_step = next(
#     step
#     for step in plan.steps
#     if step.agent == "editor_agent"
# )

# editor_history = writer_history + [
#     {
#         "step_id": writer_step.id,
#         "agent": writer_step.agent,
#         "task": writer_step.task,
#         "result": writer_result,
#     }
# ]

# editor_result = editor_agent(
#     user_request=topic,
#     step=editor_step,
#     history=editor_history,
# )

# show("Editor Agent Test", editor_result)

## 4.5 Executor Agent

In [ ]:
AGENT_REGISTRY = {
    "research_agent": research_agent,
    "writer_agent": writer_agent,
    "editor_agent": editor_agent,
}


def executor_agent(
    user_request: str,
    plan: TaskPlan,
    model: str = MODEL_NAME,
    writer_requirements: dict[str, int] | None = None,
    verbose: bool = True,
) -> list[dict[str, Any]]:
    validate_task_plan(plan, max_steps=len(plan.steps))

    history: list[dict[str, Any]] = []

    if verbose:
        print("[EXECUTOR_AGENT] Starting plan execution.")
        print(f"[EXECUTOR_AGENT] Goal: {plan.goal}")
        print(f"[EXECUTOR_AGENT] Total steps: {len(plan.steps)}")

    for position, step in enumerate(plan.steps, start=1):
        agent_function = AGENT_REGISTRY.get(step.agent)

        if verbose:
            print(
                f"[EXECUTOR_AGENT] Dispatching step {position}/"
                f"{len(plan.steps)} to {step.agent}."
            )

        try:
            if agent_function is None:
                raise ValueError(f"Unsupported agent: {step.agent!r}")

            agent_arguments = {
                "user_request": user_request,
                "step": step,
                "history": history,
                "model": model,
                "verbose": verbose,
            }

            if step.agent == "writer_agent":
                agent_arguments["report_requirements"] = writer_requirements

            result = agent_function(**agent_arguments)
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"
            history.append(
                {
                    "step_id": step.id,
                    "agent": step.agent,
                    "task": step.task,
                    "status": "failed",
                    "result": None,
                    "error": error,
                }
            )

            if verbose:
                print(f"[EXECUTOR_AGENT] Step {step.id} failed: {error}")
                print("[EXECUTOR_AGENT] Plan execution stopped.")
            break

        history.append(
            {
                "step_id": step.id,
                "agent": step.agent,
                "task": step.task,
                "status": "completed",
                "result": result,
                "error": None,
            }
        )

        if verbose:
            print(f"[EXECUTOR_AGENT] Step {step.id} completed.")

    if verbose:
        completed_steps = sum(item["status"] == "completed" for item in history)
        failed_steps = sum(item["status"] == "failed" for item in history)
        if failed_steps == 0:
            print("[EXECUTOR_AGENT] Plan execution completed.")
        print(f"[EXECUTOR_AGENT] Completed steps: {completed_steps}")
        print(f"[EXECUTOR_AGENT] Failed steps: {failed_steps}")

    return history

In [ ]:
# execution_history = executor_agent(
#     user_request=topic,
#     plan=plan,
# )

# final_report = execution_history[-1]["result"]
# show("Executor Agent Test: Final Report", final_report)

In [ ]:
def validate_final_report(
    report: str,
    min_words: int = 1200,
    max_words: int = 1800,
) -> int:
    """Validate deterministic publication requirements and return the word count."""

    if min_words < 1 or max_words < min_words:
        raise ValueError("Report word limits are invalid.")

    report = report.strip()

    if not report:
        raise RuntimeError("Final report is empty.")

    if re.search(r"(?m)^#\s+\S", report) is None:
        raise RuntimeError("Final report must contain a top-level Markdown title.")

    if re.search(r"(?im)^##\s+(references|sources)\s*$", report) is None:
        raise RuntimeError("Final report must contain a References or Sources section.")

    if re.search(r"https?://", report) is None:
        raise RuntimeError("Final report must contain source URLs.")

    word_count = len(report.split())

    if not min_words <= word_count <= max_words:
        raise RuntimeError(
            f"Final report has {word_count} words; expected "
            f"{min_words} to {max_words}."
        )

    return word_count


def run_deep_research(
    topic: str,
    output_path: str | Path = "final_report.md",
    model: str = MODEL_NAME,
    max_steps: int = 4,
    min_report_words: int = 1200,
    max_report_words: int = 1800,
    verbose: bool = True,
) -> str:
    """Plan, execute, validate, and save one deep research report."""

    if min_report_words < 1 or max_report_words < min_report_words:
        raise ValueError("Report word limits are invalid.")

    writer_requirements = {
        "min_words": min_report_words,
        "max_words": max_report_words,
    }

    plan = planner_agent(
        topic=topic,
        model=model,
        max_steps=max_steps,
        verbose=verbose,
    )
    history = executor_agent(
        user_request=topic,
        plan=plan,
        model=model,
        writer_requirements=writer_requirements,
        verbose=verbose,
    )

    failed_steps = [item for item in history if item["status"] == "failed"]

    if failed_steps:
        failed_step = failed_steps[0]
        raise RuntimeError(
            f"Deep research failed at step {failed_step['step_id']}: "
            f"{failed_step['error']}"
        )

    if len(history) != len(plan.steps):
        raise RuntimeError("Executor did not complete every planned step.")

    final_step = history[-1]

    if final_step["agent"] != "writer_agent":
        raise RuntimeError("The final completed step must use writer_agent.")

    final_report = final_step["result"]

    if not isinstance(final_report, str):
        raise TypeError("Final writer result must be a string.")

    final_report = final_report.strip()
    word_count = validate_final_report(
        final_report,
        min_words=min_report_words,
        max_words=max_report_words,
    )

    report_path = Path(output_path)
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(f"{final_report}\n", encoding="utf-8")

    if verbose:
        print(f"[DEEP_RESEARCH] Final report validated: words={word_count}")
        print(f"[DEEP_RESEARCH] Final report saved to: {report_path}")

    return final_report

In [ ]:
final_report = run_deep_research(
    "What are the recent developments in AI design patterns for improving the "
    "reliability of LLM agents?"
)
show("Final Report", final_report)